In [ ]:
from pyspark.sql import functions as F

from src.common.silver_incremental import (
    NO_TABLES_SENTINEL,
    append_silver_audit,
    build_audit_record,
    display_step_summary,
    ensure_silver_audit_table,
    filter_pending_step_rows,
    list_processing_tables,
    merge_silver_table,
    resolve_merge_keys,
    silver_audit_path,
    silver_audit_table,
    summary_from_record,
)

dbutils.widgets.text('environment', 'dev')
dbutils.widgets.text('catalog_name', 'naturapet_dev')
dbutils.widgets.text('storage_account', 'demodldb')
dbutils.widgets.text('storage_container', 'democodex')
dbutils.widgets.text('data_year', '2026')
dbutils.widgets.text('domain_name', 'comercial')

environment = dbutils.widgets.get('environment').strip().lower()
catalog = dbutils.widgets.get('catalog_name').strip()
storage_account = dbutils.widgets.get('storage_account').strip()
storage_container = dbutils.widgets.get('storage_container').strip()
data_year = dbutils.widgets.get('data_year').strip()
domain = dbutils.widgets.get('domain_name').strip().lower()
silver_schema = f'{domain}_silver'
storage_uri = f'abfss://{storage_container}@{storage_account}.dfs.core.windows.net'
silver_root = f'{storage_uri}/external/{environment}/{domain}/silver/{data_year}'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{silver_schema}`')

STEP_NAME = 'domain_normalization'
PREVIOUS_STEP = 'null_corrections'
audit_table = silver_audit_table(catalog, silver_schema)
audit_path = silver_audit_path(silver_root)
ensure_silver_audit_table(spark, audit_table, audit_path)

NORMALIZATION_COLUMNS = {
    'estado_venta': 'estado_venta_codigo',
    'estado_linea': 'estado_linea_codigo',
    'tipo_pedido': 'tipo_pedido_codigo',
    'tipo_solucion': 'tipo_solucion_codigo',
    'estado_devolucion': 'estado_devolucion_codigo',
    'medio_principal': 'medio_principal_codigo',
    'motivo_ajuste': 'motivo_ajuste_codigo',
    'estado_ajuste': 'estado_ajuste_codigo',
    'estado_orden': 'estado_orden_codigo',
    'motivo_compra': 'motivo_compra_codigo',
    'estado_stock': 'estado_stock_codigo',
    'estado_entrega': 'estado_entrega_codigo',
    'tipo_vehiculo': 'tipo_vehiculo_codigo',
    'tipo_registro': 'tipo_registro_codigo',
    'fuente_registro': 'fuente_registro_codigo',
    'tipo_presupuesto': 'tipo_presupuesto_codigo',
    'estado_sede': 'estado_sede_codigo',
    'tipo_sede': 'tipo_sede_codigo',
    'especie': 'especie_codigo',
    'unidad_medida': 'unidad_medida_codigo',
    'nivel_rotacion': 'nivel_rotacion_codigo',
    'estado_producto': 'estado_producto_codigo',
    'tipo_cliente': 'tipo_cliente_codigo',
    'genero': 'genero_codigo',
    'segmento_cliente': 'segmento_cliente_codigo',
    'tipo_mascota_principal': 'tipo_mascota_codigo',
    'nivel_fidelidad': 'nivel_fidelidad_codigo',
    'rango_edad': 'rango_edad_codigo',
    'cargo': 'cargo_codigo',
    'tipo_contrato': 'tipo_contrato_codigo',
    'estado_empleado': 'estado_empleado_codigo',
    'nivel_experiencia': 'nivel_experiencia_codigo',
    'tipo_proveedor': 'tipo_proveedor_codigo',
    'estado_proveedor': 'estado_proveedor_codigo',
    'tipo_campana': 'tipo_campana_codigo',
    'moneda': 'moneda_codigo'
}
ALLOWED_VALUES = {
    'estado_venta': ['Emitida', 'Facturada'],
    'estado_devolucion': ['Aprobada', 'En revision', 'Rechazada'],
    'estado_orden': ['Parcial', 'Pendiente', 'Recibida'],
    'estado_stock': ['Bajo', 'Normal', 'Sobrestock'],
    'estado_entrega': ['Cancelado en ruta', 'Entregado', 'Entregado tarde', 'Intento fallido']
}

def to_code(column_name):
    return F.lower(F.regexp_replace(F.regexp_replace(F.trim(F.col(column_name)), '[^A-Za-z0-9]+', '_'), '^_|_$', ''))

def normalize_table(table_name, df):
    for source_column, target_column in NORMALIZATION_COLUMNS.items():
        if source_column in df.columns:
            df = df.withColumn(target_column, to_code(source_column))
            if source_column in ALLOWED_VALUES:
                allowed = ALLOWED_VALUES[source_column]
                df = df.withColumn(f'{target_column}_valido', F.col(source_column).isin(allowed) | F.col(source_column).isNull())
    return df.withColumn('_np_silver_step', F.lit('domain_normalization')).withColumn('_np_silver_processed_ts', F.current_timestamp())

def silver_tables():
    return [row.tableName for row in spark.sql(f'SHOW TABLES IN `{catalog}`.`{silver_schema}`').collect() if not row.isTemporary]

def write_silver(df, table_name):
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    df = df.withColumn('_np_silver_step', F.lit(STEP_NAME)).withColumn('_np_silver_processed_ts', F.current_timestamp())
    merge_keys = resolve_merge_keys(table_name, df.columns)
    merge_silver_table(spark, df, target_table, target_path, merge_keys)
    return {'target_table': target_table, 'target_path': target_path, 'rows': df.count()}

summary = []
tables = list_processing_tables(spark, catalog, silver_schema)
if not tables:
    record = build_audit_record(
        environment=environment,
        domain_name=domain,
        step_name=STEP_NAME,
        table_name=NO_TABLES_SENTINEL,
        source_table=f'`{catalog}`.`{silver_schema}`',
        target_table=f'`{catalog}`.`{silver_schema}`',
        input_rows=0,
        output_rows=0,
        status='NO_DATA',
        message='No existen tablas Silver para procesar en este dominio.',
        target_path=silver_root,
    )
    append_silver_audit(spark, audit_table, audit_path, record)
    summary.append(summary_from_record(record))

for table_name in tables:
    target_table = f'`{catalog}`.`{silver_schema}`.`{table_name}`'
    target_path = f'{silver_root}/{table_name}'
    df = filter_pending_step_rows(spark.table(target_table), PREVIOUS_STEP)
    input_rows = df.count()

    if input_rows == 0:
        record = build_audit_record(
            environment=environment,
            domain_name=domain,
            step_name=STEP_NAME,
            table_name=table_name,
            source_table=target_table,
            target_table=target_table,
            input_rows=0,
            output_rows=0,
            status='NO_DATA',
            message=f'No hay filas pendientes desde {PREVIOUS_STEP}.',
            target_path=target_path,
        )
        append_silver_audit(spark, audit_table, audit_path, record)
        summary.append(summary_from_record(record))
        continue

    df = normalize_table(table_name, df)
    write_result = write_silver(df, table_name)
    record = build_audit_record(
        environment=environment,
        domain_name=domain,
        step_name=STEP_NAME,
        table_name=table_name,
        source_table=target_table,
        target_table=target_table,
        input_rows=input_rows,
        output_rows=write_result['rows'],
        status='SUCCESS',
        message=f'Filas pendientes desde {PREVIOUS_STEP} procesadas incrementalmente.',
        target_path=target_path,
    )
    append_silver_audit(spark, audit_table, audit_path, record)
    summary.append(summary_from_record(record))

display(display_step_summary(spark, summary))
